In [257]:
NAME: "Cordell Anderson"
EMAIL = "cordellanderson@arizona.edu" 

In [258]:
from typing import Iterator, Tuple, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier
from sklearn.naive_bayes import GaussianNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
import csv
from scipy.sparse import spmatrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid

In [267]:
def read_data(file_path: str) -> Tuple[Iterator[int], Iterator[str], Iterator[int]]:
    """
    Reads in the .csv data from the given filepath.
    Each row in the data contains an ID, a sentence, and optionally a label.
    If no label exists in the imported data set, the label will be set to -1.
    
    :param file_path: The path of a .csv data file, formatted as stated above.
    :return: 
        A list of three lists:
            - The IDs of the data
            - The sentences that have been read in
            - The labels for each data row
    """
    IDs = []
    sentences = []
    labels = []
    first = True
    with open(file_path, mode='r') as file:
        csvFile = csv.reader(file)
        for line in csvFile:
            if first: #skip the first line, it's the headers
                first=False
            else:
                IDs.append(line[0])
                sentences.append(line[1])
                if len(line)<3:
                    labels.append(-1)
                else:
                    labels.append(line[2])
    return IDs,sentences,labels

In [268]:
class Feature_encoder:
    def __init__(self):
        """
        Initialize the feature encoder
        """
        self.encoder = TfidfVectorizer(stop_words="english", strip_accents='unicode',ngram_range=(1,2))
        
    
    def train(self, data: Iterator[str]) -> spmatrix:
        """
        Fits the feature_encoder to the provided data.
        
        :param data: List of string sentences that the feature encoder will train on.
        :return: The feature matrix.
        """
        return self.encoder.fit_transform(data)
    
    
    def transform(self, data: Iterator[str]) -> spmatrix:
        """
        transforms the provided data into a feature matrix using this class's encoder.
        
        :param data: The data that is being transformed into a feature matrix.
        :return: The feature matrix created by transforming the given data.
        """
        return self.encoder.transform(data)

In [275]:
class Classifier:
    def __init__(self):
        """
        Initializes the classifier
        """
        #self.classifier = RidgeClassifier(tol=1e-2, solver="sparse_cg")
        # self.classifier = GaussianNB() did not work, needed full array which would take too much memory
        #self.classifier = RandomForestClassifier(max_depth=12,random_state=0)
        #self.classifier = ComplementNB(alpha=0.1)
        #self.classifier = KNeighborsClassifier(n_neighbors=100)
        #self.classifier = NearestCentroid()
        self.classifier = LogisticRegression(C=20,max_iter=1000, solver='sag')
        #self.classifier = SGDClassifier(loss="modified_huber", alpha=1e-4, n_iter_no_change=3, early_stopping=True)
    
    def train(self, train_x: spmatrix, train_y: spmatrix):
        """
        Trains the classifier using the provided input data and labels
        
        :param train_x: the feature data the model is being trained on.
        :param train_y: the target labels associated with the training data
        """
        self.classifier.fit(train_x, train_y)
    
        
    def predict(self, test_x: spmatrix) -> Iterator[int]:
        """
        Makes a prediction on the data passed in to the function. Make sure you have run the train() method first!
        
        :param data: The data that predictions are going to be made on.
        :return: A list of predictions made by the classifier for the given data.
        """
        return self.classifier.predict(test_x)

In [270]:
def prepare_data(
    encoder,
    train_file: str,
    test_file: str="",
    PRACTICE: bool=True
) -> Tuple[spmatrix, spmatrix, Iterator[int], Iterator[int], Iterator[int]]:
    """
    Reads in the test/train data using the provided file paths. If PRACTICE is True, creates a test set
    in a train/test split from the training data.
    
    :param encoder: the class instance of whatever method will be used to encode the read in data
    as a feature matrix.
    :param train_file: The file path to the location of the training data.
    :param test_file: The file path to the location of the testing data. Defaults to empty.
    :param PRACTICE: Boolean flag to determine if this is a practice test that should create its own test data,
    or if there is real, unlabeled data to be read in.
    return: 
        A tuple containing:
            - the feature matrix for the training data
            - the feature matrix for the testing data
            - the labels for the training data
            - the labels for the testing data (this will be empty if not practicing)
            - the IDs associated with the test data (this will be empty if practicing)
    """
    
    X_train = X_test = y_train = y_test = IDs = None
    _, train_x_raw, train_labels = read_data(train_file)
    
    if PRACTICE:
        X_train, X_test, y_train, y_test = train_test_split(
        train_x_raw, train_labels, 
        test_size=0.2,    # 20% for testing
        random_state=42,  # Ensures reproducibility
        shuffle=True      # Mixes data before splitting
        )
    else:
        X_train = train_x_raw
        y_train = train_labels
        IDs, X_test, _ = read_data(test_file)
        
    X_train_encoded = encoder.train(X_train)
    X_test_encoded = encoder.transform(X_test)
    
        
    return X_train_encoded, X_test_encoded, y_train, y_test, IDs

In [271]:
def save(path: str, predictions: Iterator[int], IDs: Iterator[int]):
    """
    Creates a .csv file at the provided location containing the predictions and IDs passed in.
    
    :param path: The file path and name the data should be saved to. This should end in <FILENAME>.csv
    :param predictions: A list of predictions made about the data
    :param IDs: A list of IDs associated with each prediction.
    """
    
    if len(predictions) != len(IDs): # Make sure the two lists are the same length.
        print("Length mismatch.")
        return
    
    output_data = []
    for i in range(len(IDs)):
            output_data.append({"ID":IDs[i],"LABEL":predictions[i]})
        
    with open(path,'w', newline='') as file:
        headers = ["ID","LABEL"]
        writer = csv.DictWriter(file, fieldnames=headers)
        writer.writeheader()
        writer.writerows(output_data)

In [272]:
def compare(predictions: Iterator[int], values: Iterator[int]) -> str:
    """
    Compares the two input lists, and calculates the percent that matched.
    
    :param predictions: The list of predictions made by the classifier
    :param values: The correct labels for each datapoint.
    
    :return: A string reporting the results of the comparison"""
    total=0
    correct=0
    for i in range(len(predictions)):
        total +=1
        if predictions[i]==values[i]:
            correct+=1
    output = f"Correct: {correct}, Total: {total}, Accuracy: {(correct/total)*100}%"
    return output

In [279]:
PRACTICE = False

fe = Feature_encoder()
x_train, x_test, y_train, y_test, IDs = prepare_data(fe, "data/train.csv", "data/test.csv",PRACTICE)

In [280]:
cf = Classifier()
cf.train(x_train, y_train)
predictions = cf.predict(x_test)

if PRACTICE:
    print(compare(predictions, y_test))
else:
    save("data/output.csv",predictions,IDs)

In [223]:
# Ridge classifier with TFIDF feature encoder: 89.9601%
# Random forest classifier with TFIDF feature encoder: 59.2363%
# ComplementNB classifier TFIDF feature encoder: 84.307%
# KNN / TFIDF: 46.153
# nearestCentroid / TFIDF: 80.922
# LogisticRegression / TFIDF: 91.936%
# LogisticRegression(sag) / TFIDF: 92.6265%
# SGDClassifer hinge / TFIDF: 90.6996
# SGDClassifer modified_huber / TFIDF: 91.7804
